## 📊 Cómo Comparar Modelos usando las Predicciones Guardadas

A continuación se muestra cómo cargar y comparar las predicciones de diferentes modelos:

### Cargar Predicciones

```python
import pickle
import numpy as np

# Cargar predicciones del modelo actual
with open('predictions_resnet50_cv5.pkl', 'rb') as f:
    results_model1 = pickle.load(f)

# O cargar desde formato NumPy (más rápido)
data = np.load('predictions_resnet50_cv5.npz', allow_pickle=True)
predictions_1 = data['predictions']
true_labels_1 = data['true_labels']
probabilities_1 = data['probabilities']
class_names = data['class_names']
```

### Comparar Curvas ROC de Múltiples Modelos

El ejemplo siguiente muestra cómo comparar este modelo con otros modelos.

In [ ]:
# Función para comparar curvas ROC de múltiples modelos
def compare_models_roc(model_files, model_names=None, save_plot=False):
    """
    Compara las curvas ROC de múltiples modelos
    
    Args:
        model_files: Lista de rutas a archivos .pkl con predicciones
        model_names: Lista de nombres para cada modelo (opcional)
        save_plot: Si es True, guarda la imagen
    """
    from sklearn.preprocessing import label_binarize
    from sklearn.metrics import roc_curve, auc
    
    if model_names is None:
        model_names = [f"Modelo {i+1}" for i in range(len(model_files))]
    
    plt.figure(figsize=(12, 9))
    
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    for idx, (model_file, model_name) in enumerate(zip(model_files, model_names)):
        # Cargar predicciones
        with open(model_file, 'rb') as f:
            results = pickle.load(f)
        
        y_true = results['true_labels']
        y_prob = np.array(results['probabilities'])
        num_classes = results['num_classes']
        
        # Binarizar etiquetas
        y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
        
        # Calcular curva ROC micro-average
        fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
        roc_auc = auc(fpr, tpr)
        
        # Graficar
        plt.plot(fpr, tpr, color=colors[idx % len(colors)], lw=2.5,
                label=f'{model_name} (AUC = {roc_auc:.3f})')
        
        print(f"✅ {model_name}:")
        print(f"   AUC: {roc_auc:.4f}")
        print(f"   Precisión: {results['global_accuracy']:.4f}")
        print(f"   Muestras: {results['total_samples']}")
        print()
    
    # Línea diagonal
    plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Aleatorio (AUC = 0.5)')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos', fontsize=12)
    plt.ylabel('Tasa de Verdaderos Positivos', fontsize=12)
    plt.title('Comparación de Curvas ROC - Múltiples Modelos', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_plot:
        plt.savefig('roc_comparison.png', dpi=300, bbox_inches='tight')
        print("💾 Gráfico guardado como 'roc_comparison.png'")
    
    plt.show()

# Ejemplo de uso (descomenta para ejecutar):
# compare_models_roc(
#     model_files=['predictions_resnet50_cv5.pkl', 'predictions_otro_modelo.pkl'],
#     model_names=['ResNet50 CV5', 'Otro Modelo'],
#     save_plot=True
# )


In [ ]:
# Función para comparar métricas detalladas entre modelos
def compare_models_metrics(model_files, model_names=None):
    """
    Compara métricas detalladas de múltiples modelos
    
    Args:
        model_files: Lista de rutas a archivos .pkl con predicciones
        model_names: Lista de nombres para cada modelo (opcional)
    """
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    if model_names is None:
        model_names = [f"Modelo {i+1}" for i in range(len(model_files))]
    
    comparison_data = []
    
    for model_file, model_name in zip(model_files, model_names):
        # Cargar predicciones
        with open(model_file, 'rb') as f:
            results = pickle.load(f)
        
        y_true = results['true_labels']
        y_pred = results['predictions']
        
        # Calcular métricas
        metrics = {
            'Modelo': model_name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, average='weighted'),
            'Recall': recall_score(y_true, y_pred, average='weighted'),
            'F1-Score': f1_score(y_true, y_pred, average='weighted'),
            'AUC Micro': results.get('roc_auc_micro', 'N/A'),
            'AUC Macro': results.get('roc_auc_macro', 'N/A'),
            'Muestras': results['total_samples']
        }
        
        comparison_data.append(metrics)
    
    # Crear DataFrame
    df_comparison = pd.DataFrame(comparison_data)
    
    print("\n" + "="*80)
    print("📊 COMPARACIÓN DETALLADA DE MODELOS")
    print("="*80)
    print(df_comparison.to_string(index=False))
    print("="*80 + "\n")
    
    # Visualización
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors = ['steelblue', 'coral', 'seagreen', 'orchid']
    
    for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
        ax = axes[idx // 2, idx % 2]
        values = df_comparison[metric].values
        bars = ax.bar(range(len(model_names)), values, color=color, alpha=0.7, edgecolor='black')
        
        # Añadir valores sobre las barras
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}',
                   ha='center', va='bottom', fontweight='bold')
        
        ax.set_ylabel(metric, fontsize=12)
        ax.set_title(f'Comparación: {metric}', fontweight='bold')
        ax.set_xticks(range(len(model_names)))
        ax.set_xticklabels(model_names, rotation=45, ha='right')
        ax.set_ylim([0, 1.1])
        ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    return df_comparison

# Ejemplo de uso (descomenta para ejecutar):
# df_comp = compare_models_metrics(
#     model_files=['predictions_resnet50_cv5.pkl', 'predictions_otro_modelo.pkl'],
#     model_names=['ResNet50 CV5', 'Otro Modelo']
# )


In [ ]:
# Función para inspeccionar el contenido de un archivo de predicciones
def inspect_predictions_file(filename):
    """
    Muestra información detallada de un archivo de predicciones guardado
    
    Args:
        filename: Ruta al archivo .pkl
    """
    with open(filename, 'rb') as f:
        data = pickle.load(f)
    
    print("\n" + "="*70)
    print(f"📄 INFORMACIÓN DEL ARCHIVO: {filename}")
    print("="*70)
    
    print("\n🔍 Contenido del archivo:")
    for key in data.keys():
        value = data[key]
        if isinstance(value, (list, np.ndarray)):
            print(f"   • {key}: {type(value).__name__} con {len(value)} elementos")
        elif isinstance(value, dict):
            print(f"   • {key}: diccionario con {len(value)} claves")
        else:
            print(f"   • {key}: {value}")
    
    print("\n📊 Información del Modelo:")
    print(f"   • Nombre: {data.get('model_name', 'N/A')}")
    print(f"   • Arquitectura: {data.get('model_architecture', 'N/A')}")
    print(f"   • Fecha: {data.get('date', 'N/A')}")
    
    print("\n📈 Métricas:")
    print(f"   • Accuracy global: {data.get('global_accuracy', 'N/A'):.4f}")
    print(f"   • AUC Micro: {data.get('roc_auc_micro', 'N/A'):.4f}")
    print(f"   • AUC Macro: {data.get('roc_auc_macro', 'N/A'):.4f}")
    print(f"   • Muestras totales: {data.get('total_samples', 'N/A')}")
    print(f"   • Número de clases: {data.get('num_classes', 'N/A')}")
    
    if 'fold_results' in data:
        print("\n🔄 Información de Cross-Validation:")
        print(f"   • Número de folds: {data.get('cv_folds', 'N/A')}")
        print(f"   • Precisión media: {data.get('mean_accuracy', 'N/A'):.4f}")
        print(f"   • Desviación estándar: {data.get('std_accuracy', 'N/A'):.4f}")
        print("\n   Resultados por fold:")
        for fold in data['fold_results']:
            print(f"      - Fold {fold['fold']}: {fold['best_val_acc']:.4f}")
    
    print("\n" + "="*70 + "\n")
    
    return data

# Ejemplo de uso (descomenta para ejecutar):
# data = inspect_predictions_file('predictions_resnet50_cv5.pkl')
